# บทเรียน 11 - โปรโตคอล ตัวแทนต่อ-ตัวแทน (A2A)


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## โปรโตคอล A2A คืออะไร?

โปรโตคอล **เอเยนต์ต่อเอเยนต์ (A2A)** เป็นมาตรฐานแบบเปิดที่ช่วยให้เอเยนต์ AI สามารถสื่อสาร ค้นพบกันและกัน และทำงานร่วมกัน — แม้ว่าจะถูกพัฒนาบนเฟรมเวิร์กที่แตกต่างกันหรือโฮสต์โดยบริการที่ต่างกัน

แนวคิดหลัก:

- **Discovery** – เอเยนต์เผยแพร่ *Agent Card* ที่อธิบายความสามารถของพวกเขา ทำให้เอเยนต์อื่น (หรือผู้ประสานงาน) สามารถค้นหาผู้เชี่ยวชาญที่เหมาะสมสำหรับงานได้อย่างง่ายดาย
- **Message Passing** – เอเยนต์แลกเปลี่ยนข้อความที่มีโครงสร้างผ่านโปรโตคอลร่วม ดังนั้นคำขอจากเอเยนต์หนึ่งจึงสามารถถูกเข้าใจและปฏิบัติได้โดยอีกเอเยนต์หนึ่ง โดยไม่คำนึงถึงการดำเนินงานภายในของมัน
- **Task Lifecycle** – A2A นิยามสถานะเช่น *submitted*, *working*, *completed*, และ *failed* ซึ่งให้ผู้ประสานงานมองเห็นได้อย่างครบถ้วนว่างานที่ได้รับมอบหมายกำลังดำเนินไปอย่างไร

ในบทเรียนนี้ เราจำลองการทำงานร่วมกันแบบ A2A โดยเชื่อมต่อเอเยนต์ท่องเที่ยวเฉพาะทางสามตัวเข้ากับเวิร์กโฟลว์ที่แต่ละเอเยนต์มีส่วนร่วมด้วยความเชี่ยวชาญของตนและส่งผลลัพธ์ไปยังเอเยนต์ถัดไป


## การสร้างตัวแทนท่องเที่ยวเฉพาะทาง


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## การทำงานร่วมกันของหลายเอเจนต์ผ่านเวิร์กโฟลว์

เราเชื่อมต่อเอเจนต์ทั้งสามเข้าเป็นเวิร์กโฟลว์แบบลำดับที่สะท้อนการส่งข้อความ A2A:

1. **CurrencyExchangeAgent** ได้รับคำขอจากผู้ใช้และให้คำแนะนำเกี่ยวกับสกุลเงิน.
2. **ActivityPlannerAgent** ได้รับบริบทที่ขยายแล้วและเพิ่มคำแนะนำเกี่ยวกับกิจกรรม.
3. **TravelManagerAgent** สังเคราะห์อินพุตทั้งสองเป็นบทสรุปการเดินทางฉบับสุดท้าย.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## การทำความเข้าใจ A2A ในสภาพแวดล้อมการผลิต

ในสภาพแวดล้อมการผลิต โปรโตคอล A2A เปิดทางให้เกิดสถานการณ์ข้ามบริการที่ทรงพลัง:

| Capability | Description |
|---|---|
| **การทำงานร่วมข้ามเฟรมเวิร์ก** | เอเจนต์ที่สร้างด้วยเฟรมเวิร์กหนึ่งสามารถมอบหมายงานให้เอเจนต์ที่สร้างด้วยเฟรมเวิร์กอื่น ๆ ที่เป็นไปตามข้อกำหนด A2A ได้ ช่วยให้สามารถทำงานร่วมกันข้ามองค์กรได้อย่างแท้จริง. |
| **ขอบเขตของบริการ** | เอเจนต์สามารถอยู่ในไมโครเซอร์วิส แถบภูมิภาคคลาวด์ หรือแม้กระทั่งองค์กรที่ต่างกัน ขณะเดียวกันก็สามารถทำงานร่วมกันได้อย่างราบรื่น. |
| **การค้นหาแบบไดนามิก** | ออเคสเตรเตอร์สามารถสอบถามรีจิสตรี Agent Card ขณะรันไทม์เพื่อค้นหาผู้เชี่ยวชาญที่เหมาะสมที่สุดสำหรับงานย่อยหนึ่ง ๆ. |
| **การสตรีม & การแจ้งเตือนแบบพุช** | A2A รองรับ Server-Sent Events (SSE) สำหรับอัปเดตความคืบหน้าแบบเรียลไทม์ และการแจ้งเตือนแบบพุชสำหรับงานที่ใช้เวลานาน. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## สรุป

ในบทเรียนนี้คุณได้เรียนรู้:

1. **โปรโตคอล A2A คืออะไร** — มาตรฐานเปิดสำหรับการค้นหาระหว่างเอเจนต์, การส่งข้อความ,
   และการจัดการงาน.
2. **วิธีการสร้างเอเจนต์เฉพาะทาง** — เอเจนต์แลกเปลี่ยนสกุลเงิน, เอเจนต์วางแผนกิจกรรม,
   และตัวประสานงาน Travel Manager.
3. **วิธีการเชื่อมเอเจนต์เข้ากับเวิร์กโฟลว์** — ใช้ `WorkflowBuilder` เพื่อจำลองการส่งข้อความแบบ
   ต่อเนื่องระหว่างเอเจนต์.
4. **A2A ทำงานอย่างไรในการใช้งานจริง** — ช่วยให้สามารถร่วมมือกันข้ามเฟรมเวิร์กและข้ามบริการ
   ด้วยการค้นหาแบบไดนามิกและการอัปเดตแบบสตรีมมิ่ง.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ข้อจำกัดความรับผิดชอบ:
เอกสารฉบับนี้ได้รับการแปลด้วยบริการแปลภาษาโดยปัญญาประดิษฐ์ Co-op Translator (https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามให้การแปลถูกต้อง โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความคลาดเคลื่อนได้ เอกสารต้นฉบับในภาษาเดิมควรถูกถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่มีความสำคัญ ขอแนะนำให้ใช้การแปลโดยนักแปลมืออาชีพเป็นการตรวจสอบเพิ่มเติม เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดใดๆ ที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
